In [1]:
import sys 
sys.path.append("../../")

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_score, recall_score, accuracy_score, confusion_matrix, f1_score, roc_auc_score, matthews_corrcoef
import muon as mu
import scirpy as ir
from dextrademixer.model import DextraDemixer, BEAMT, ITRAP,  ICON

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="mudata")
# mu.set_options(pull_on_update=True) 

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

/Users/yang.an/.local/share/mamba/envs/dextrademixer_minimal/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/yang.an/.local/share/mamba/envs/dextrademixer_minimal/lib/python3.13/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [2]:
%load_ext autoreload
%autoreload 2

# Set up

In [3]:
# ground truth and HLA per dextramer
dex_map ={"HLA-A*0201": "DexA", "HLA-C*0702":"DexC", "HLA-B*0801":"DexN", "HLA-B*0702":"DexB"}
dex_map_rev ={"DexA":"HLA-A*0201", "DexC":"HLA-C*0702", "DexN":"HLA-B*0801", "DexB":"HLA-B*0702"}
dex_info = {
    'DexA': {'ground_truth': 'NLV_true', 'HLA-match': 'D2'},
    'DexB': {'ground_truth': 'TPR_true', 'HLA-match': 'D1'},
    'DexC': {'ground_truth': 'CRV_true', 'HLA-match': 'D1'},
    
}

In [4]:
def eval_results(y_true, y_pred, p_pred=False, adata=None, method=None):
    mask = ~np.isnan(y_true)
    tn, fp, fn, tp = confusion_matrix(y_true[mask], y_pred[mask]).ravel()
    tpr = tp / (tp + fn)
    fdr = fp / (tp + fp)

    results = pd.Series()
    if np.any(p_pred):
        results['roc_auc'] = roc_auc_score(y_true[mask].astype(int), p_pred[mask])
        results['pr_auc'] = average_precision_score(y_true[mask].astype(int), p_pred[mask])
    else:
        results['roc_auc'], results['pr_auc'] = np.nan, np.nan
    results['f1'] = f1_score(y_true[mask].astype(int), y_pred[mask])
    results['precision'] = precision_score(y_true[mask].astype(int), y_pred[mask])
    results['recall'] = recall_score(y_true[mask].astype(int), y_pred[mask])
    results['accuracy'] = accuracy_score(y_true[mask].astype(int), y_pred[mask])
    results['mcc'] = matthews_corrcoef(y_true[mask].astype(int), y_pred[mask])
    results['fdr'] = fdr

    if adata:
        adata.obs[method] = pd.Categorical(['excluded' for i in range(adata.n_obs)], categories = ['TP','TN','FN','FP', 'excluded'])
        adata.obs.loc[mask,method] = np.where(y_true[mask]==1, np.where(y_pred[mask]==1, 'TP', 'FN'), np.where(y_pred[mask]==1, 'FP', 'TN'))

    return results

In [5]:
# Helpers

def run_BEAMT(
    mdat,
    pmhc_key="HLA-A*0201",
    gex_key="rio",
    ir_key="airr",
    neg_ctrl_key="HLA-B*0801",
    threshold=0.5,
    target_fdr=None
):
    mask = ~np.isnan(mdat.obs[dex_info[dex_map[pmhc_key]]['ground_truth']].values)
    mdat.obs[[f'BEAM_{dex_map[pmhc_key]}',f'BEAM_{dex_map[pmhc_key]}_p']] = np.nan
    
    mixer = BEAMT()
    mixer.preprocess_model_data(mdat[mask], pmhc_key=pmhc_key, gex_key=gex_key, neg_ctrl_key=neg_ctrl_key, ir_key=ir_key)
    mixer.fit()
    p_pred, assignment = mixer.predict_posterior_class(target_fdr=target_fdr, threshold=threshold)
    mdat.obs.loc[mask,f'BEAM_{dex_map[pmhc_key]}'] = assignment.__array__()
    mdat.obs.loc[mask,f'BEAM_{dex_map[pmhc_key]}_p'] = p_pred.__array__()
    return mdat


def run_dextrademixer(
    mdat,
    pmhc_key="HLA-A*0201",
    gex_key="rio",
    ir_key="airr",
    neg_ctrl_key="HLA-B*0801",
    clone_id_key='clone_id',
    threshold=0.5,
    target_fdr=None
):
    mask = ~np.isnan(mdat.obs[dex_info[dex_map[pmhc_key]]['ground_truth']].values)
    mdat.obs[[f'dextrademixer_{dex_map[pmhc_key]}',f'dextrademixer_{dex_map[pmhc_key]}_p']] = np.nan

    # hyper parameters
    seed, alpha_offset, hyperprior, use_size_factor, outlier_threshold = 42, 5, 1.0, True, None
    opt_params = {"maxiter": 1000, "nof_inits": 10, "adam": {"init_value": 3e-1, "end_value": 3e-3, "decay_rate": 0.995, "transition_steps": 1}, }

    # run model
    mixer = DextraDemixer(
        model_type='mixturemodelkmeans', mode='I', alpha_model='overdispersion',
        model_config={"overdispersion_scale_prior": hyperprior, "alpha_offset": alpha_offset}
    )
    mixer.preprocess_model_data(
        mdat[mask], pmhc_key=pmhc_key, gex_key=gex_key, neg_ctrl_key=neg_ctrl_key,
        ir_clone_key=None, outlier_threshold=outlier_threshold, use_size_factor=use_size_factor
    )
    mixer.fit_svi(guide='normal', svi_config=opt_params, nof_inits=opt_params["nof_inits"], rng_key=42, y_true=None)
    samples = mixer.get_posterior_samples()
    p_pred, assignment = mixer.predict_posterior_class(threshold=threshold, clonotype_median_p=True, clone_id=mdat.obs.loc[mask,f'{ir_key}:{clone_id_key}'])
    mdat.obs.loc[mask,f'DextraDemixer_{dex_map[pmhc_key]}'] = assignment.__array__()
    mdat.obs.loc[mask,f'DextraDemixer_{dex_map[pmhc_key]}_p'] = p_pred.__array__()
    return mdat



# Prepare data

Please download the data and scripts from [https://zenodo.org/records/15423937](https://zenodo.org/records/15423937) and follow the `Part2_CMV-Dex_Multiome.ipynb` notebook to combine `304_CMV_airr_merged.tsv` and `mudata_fromseurat.h5mu` files into `mudata_scanpy.h5mu`.

In [6]:
# needed to do
# with h5py.File("../../data/mudata_scanpy_new.h5mu", "r+") as f:
#    del f["mod/airr/uns/ir_dist_VDJDB_aa_identity/params/cutoff"]
#    del f["mod/airr/X"]
#    del f["mod/airr/uns/ir_dist_nt_identity/params/cutoff"]

# because this anndata version can't read elements with value ''null''
mdata = mu.read("../../data/mudata_scanpy.h5mu")
mdata.obs[['VDJ_1_umi_count','VJ_1_umi_count','VDJ_2_umi_count','VJ_2_umi_count']] = ir.get.airr(mdata, airr_variable=['umi_count'], chain=['VDJ_1', 'VJ_1', 'VDJ_2', 'VJ_2']).fillna(0)
mdata.pull_obs()

mdata

/Users/yang.an/.local/share/mamba/envs/dextrademixer_minimal/lib/python3.13/site-packages/anndata/utils.py:362: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  warnings.warn(msg, category, stacklevel=stacklevel)
/var/folders/l3/n75m9_cn5lzbx_tm88xrzhmhcfmwpj/T/ipykernel_61004/2708715915.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mdata.obs[['VDJ_1_umi_count','VJ_1_umi_count','VDJ_2_umi_count','VJ_2_umi_count']] = ir.get.airr(mdata, airr_variable=['umi_count'], chain=['VDJ_1', 'VJ_1', 'VDJ_2', 'VJ_2']).fillna(0)


MuData object with n_obs × n_vars = 9229 × 20154
  obs:	'VDJ_1_umi_count', 'VJ_1_umi_count', 'VDJ_2_umi_count', 'VJ_2_umi_count'
  uns:	'Cell_colors', 'Dex_colors', 'Sample_colors', 'cartridge_colors', 'cluster_colors', 'pca'
  obsm:	'AB', 'HTO', 'HTOflex', 'RNA', 'RiO', 'RiO_bg', 'X_pca', 'X_umap', 'X_wnn.umap', 'wnn.umap'
  varm:	'AB', 'HTO', 'HTOflex', 'PCs', 'RNA', 'RiO', 'RiO_bg'
  obsp:	'wknn', 'wsnn'
  7 modalities
    airr:	2835 x 0
      obs:	'receptor_type', 'receptor_subtype', 'chain_pairing', 'cluster', 'Dex', 'clone_id', 'clone_id_size', 'antigen.species', 'antigen.gene'
      uns:	'chain_indices', 'clone_id', 'clonotype_network', 'ir_dist_VDJDB_aa_identity', 'ir_dist_nt_identity', 'ir_query_VDJDB_aa_identity', 'scirpy_version'
      obsm:	'X_clonotype_network', 'airr', 'chain_indices'
    rna:	9229 x 20107
      obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_HTO', 'nFeature_HTO', 'nCount_HTOflex', 'nFeature_HTOflex', 'nReads_ALL', 'nReads_RNA', 'percent.mito', 'meanCount_RNA', 'meanCount_HTO', 'meanCount_HTOflex', 'HTOflex_maxID', 'HTOflex_secondID', 'HTOflex_margin', 'HTOflex_classification', 'HTOflex_classification.global', 'hash.ID', 'hash.ID.flex', 'HTO_maxID', 'HTO_secondID', 'HTO_margin', 'HTO_classification', 'HTO_classification.global', 'tag.no', 'Sample', 'SpikeIn.pct', 'Background', 'Background.Donor', 'Background.pct', 'flex.tag.no', 'Cell', 'Type', 'Donor', 'Donor.long', 'Origin', 'Specificity', 'peptide.stim', 'CMV.peptide', 'Peptide.Seq', 'HLA.A.02.01', 'HLA.B.07.02', 'HLA.C.07.02', 'nCount_AB', 'nFeature_AB', 'nCount_RiO', 'nFeature_RiO', 'cartridge', 'S.Score', 'G2M.Score', 'Phase', 'RNA_snn_res.0.5', 'seurat_clusters', 'nCount_SCT', 'nFeature_SCT', 'SCT.weight', 'AB.weight', 'wsnn_res.0.5', 'wsnn_res.0.5_anno', 'wsnn_res.0.5_old', 'wsnn_res.0.52', 'wsnn_res.0.52_anno', 'nCount_RiO_bg', 'nFeature_RiO_bg', 'cell_id', 'DexA', 'DexB', 'DexC', 'Dex', 'flex_tag_label', 'cluster', 'category'
      var:	'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable', 'highly_variable'
      uns:	'pca'
      obsm:	'X_pca', 'wnn.umap'
      varm:	'PCs'
      layers:	'counts', 'data'
    ab:	9229 x 31
      var:	'highly_variable'
      uns:	'apca'
      obsm:	'X_apca'
      varm:	'apca'
      layers:	'counts', 'data'
    rio:	9229 x 4
      layers:	'counts'
    rio_bg:	9229 x 3
      layers:	'counts', 'data'
    hto:	9229 x 4
      layers:	'counts'
    hto_flex:	9229 x 5
      layers:	'counts'

In [7]:
# These are the four samples: Donor 1 & 2, each with 5 or 50% theoretical spike-in 
# over all 3 clones (detected is less)
mdata.obs['rna:Sample'].value_counts(dropna=False)

rna:Sample
05-95-D2    2754
05-95-D1    2348
50-50-D1    2161
50-50-D2    1966
Name: count, dtype: int64

In [8]:
# Spike-in clones are True, HLA-mismatched BG are False (BG_D1 False for NLV, BG_D2 False for CRV)
# HLA-matched BG are nan because we don't know if there are some clonotypes that may bind
mdata.obs['NLV_true'] = np.where(mdata.obs["rna:flex_tag_label"] == 'NLV_Clone', 1, np.where(mdata.obs["rna:flex_tag_label"] =='BG_D1', np.nan, 0))
mdata.obs['CRV_true'] = np.where(mdata.obs["rna:flex_tag_label"] == 'CRV_Clone', 1, np.where(mdata.obs["rna:flex_tag_label"] == 'BG_D2', np.nan, 0))
mdata.obs['TPR_true'] = np.where((mdata.obs["rna:flex_tag_label"] == 'TPR_enriched') & (mdata.obs['rna:wsnn_res.0.5_anno'] == 'TPR_enriched_CD8_other'), 1, np.where(mdata.obs["rna:flex_tag_label"] == 'BG_D2', np.nan, 0))

In [9]:
# dataset with cells with clone id
mdata_wTCR = mdata[mdata['airr'].obs_names].copy()

# set unique clone id's to cells without TCR
clone_id = mdata.obs['airr:clone_id'].astype(float).copy()
max_id = clone_id.max()
num_nans = clone_id.isna().sum()
clone_id[clone_id.isna()] = np.arange(max_id, max_id+num_nans)
mdata.obs[f'airr:clone_id'] = clone_id.astype(str).values

# Run methods

In [10]:
for pmhc in ['HLA-A*0201', 'HLA-C*0702', 'HLA-B*0702']:
    print(pmhc)
    mdata = run_dextrademixer(mdata, pmhc_key=pmhc)
    mdata_wTCR = run_dextrademixer(mdata_wTCR, pmhc_key=pmhc)
    
    # BEAMT
    mdata = run_BEAMT(mdata, pmhc_key=pmhc)
    mdata_wTCR = run_BEAMT(mdata_wTCR, pmhc_key=pmhc)

HLA-A*0201


100%|██████████| 1000/1000 [00:06<00:00, 158.94it/s, avg. loss [991-1000]: 14035.0157]


HLA-C*0702


100%|██████████| 1000/1000 [00:06<00:00, 164.13it/s, avg. loss [991-1000]: 14446.8682]


HLA-B*0702


100%|██████████| 1000/1000 [00:05<00:00, 168.21it/s, avg. loss [991-1000]: 14524.5620]


#### icon and itrap work better using all dextramers at once

In [11]:
icon_predictions = ICON.icon_assign_pmhc(
    mdata_wTCR, 
    ir_clone_key='airr:clone_id',
    neg_ctrl_key='HLA-B*0801',
    threshold=0,
    bg_noise=None,
    bg_noise_quantile=0.975,
    dex_key='rio',
)
icon_predictions = pd.DataFrame(icon_predictions, index=mdata_wTCR.obs_names, columns=['ICON_DexA','ICON_DexC','ICON_DexB']).astype(int)
icon_predictions.head(3)

,ICON_DexA,ICON_DexC,ICON_DexB
GAGGTGCTAAACTCATTGGATCTCTTA_1,0,0,1
ATTAAGTGCACTTCGAGCACGAACCAG_1,1,0,0
TGGCTCAGAACCAGGTGTTCCGTCTTA_1,0,0,1


In [12]:
icon_predictions_faithful = ICON.icon_assign_pmhc(
    mdata_wTCR, 
    ir_clone_key='airr:clone_id',
    neg_ctrl_key='HLA-B*0801',
    threshold=0,
    bg_noise=None,
    bg_noise_quantile=0.975,
    dex_key='rio',
    faithful=True
)
icon_predictions_faithful = pd.DataFrame(icon_predictions_faithful, index=mdata_wTCR.obs_names, columns=['ICON-code_DexA','ICON-code_DexC','ICON-code_DexB']).astype(int)
icon_predictions_faithful.head(3)

,ICON-code_DexA,ICON-code_DexC,ICON-code_DexB
GAGGTGCTAAACTCATTGGATCTCTTA_1,0,0,1
ATTAAGTGCACTTCGAGCACGAACCAG_1,0,0,0
TGGCTCAGAACCAGGTGTTCCGTCTTA_1,0,0,1


In [13]:
mixer = ITRAP.ITRAP(filters=['opt_thr', 'specificity_multiplets'])
mixer.preprocess_model_data(
    mdata_wTCR, neg_ctrl_key='HLA-B*0801', ir_key='airr', ir_clone_key='clone_id', dex_key='rio', 
    umi_cols_TRA=['VJ_1_umi_count','VJ_2_umi_count'], umi_cols_TRB=['VDJ_1_umi_count','VDJ_2_umi_count'],
)
_ = mixer.assign_pmhc(mdata_wTCR)
itrap_predictions = mdata_wTCR.obsm['itrap_pMHC_assignment'].rename(columns=dex_map)
itrap_predictions.columns = 'ITRAP_' + itrap_predictions.columns
itrap_predictions.head(3)

Model has not been fit yet. Finding optimal thresholds...


/Users/yang.an/PhD/DextraDemixer/experiments/Gemuend_CMV/../../dextrademixer/model/ITRAP.py:241: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ct_pep = ct_pep.groupby(self.ir_clone_key, observed=True).apply(self._calculate_expected_target).to_frame()


,ITRAP_DexA,ITRAP_DexB,ITRAP_DexN,ITRAP_DexC
GAGGTGCTAAACTCATTGGATCTCTTA_1,0,0,1,0
ATTAAGTGCACTTCGAGCACGAACCAG_1,1,0,0,0
TGGCTCAGAACCAGGTGTTCCGTCTTA_1,0,0,1,0


In [14]:
mdata_wTCR.obs = pd.concat([mdata_wTCR.obs,icon_predictions,icon_predictions_faithful,itrap_predictions],axis=1)

# Evaluate and visualize

### Evaluate

In [15]:
methods = {'(has TCR)': ['DextraDemixer','BEAM', 'ICON', 'ITRAP', 'ICON-code'], '(all cells)': ['DextraDemixer','BEAM']}
results = []
df_setting = {}
for available_TCR in [True,False]:
    for spike_in in ['all','50-50','05-95']:
        for dextramer in ['DexA', 'DexC']:
            subsample = '(has TCR)' if available_TCR else '(all cells)'
            sample = f"{spike_in}-{dex_info[dextramer]['HLA-match']}"
            if not available_TCR:
                adata = mdata.copy()
            else:
                adata = mdata_wTCR.copy()
            if spike_in != 'all':
                adata = adata[adata.obs["rna:Sample"] == sample]
                            
            for i, method in enumerate(methods[subsample]): 
                # get predictions
                ytrue = adata.obs[dex_info[dextramer]['ground_truth']].values
                ypred = adata.obs[f'{method}_{dextramer}'].values
                ppred = adata.obs[f'{method}_{dextramer}_p'] if f'{method}_{dextramer}_p' in adata.obs.columns else False
    
                # compute metrics
                slabel = spike_in.replace('50-50', '50%').replace('05-95', '5%')
                setting = f'{dextramer} {slabel} {subsample}'
                method_results = {'dex_key': dextramer, 'model_config': method, 'sample': sample, 'tcr_info_sequenced_only': available_TCR, 'setting': setting, }
                metrics = eval_results(ytrue, ypred, ppred, adata, method=method)
                method_results.update(metrics)
                results.append(method_results)


/var/folders/l3/n75m9_cn5lzbx_tm88xrzhmhcfmwpj/T/ipykernel_61004/1143643804.py:21: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[method] = pd.Categorical(['excluded' for i in range(adata.n_obs)], categories = ['TP','TN','FN','FP', 'excluded'])
/var/folders/l3/n75m9_cn5lzbx_tm88xrzhmhcfmwpj/T/ipykernel_61004/1143643804.py:21: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[method] = pd.Categorical(['excluded' for i in range(adata.n_obs)], categories = ['TP','TN','FN','FP', 'excluded'])
/var/folders/l3/n75m9_cn5lzbx_tm88xrzhmhcfmwpj/T/ipykernel_61004/1143643804.py:21: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[method] = pd.Categorical(['excluded' for i in range(adata.n_obs)], categories = ['TP','TN','FN','FP', 'excluded'])
/var/folders/l3/n75m9_cn5lzbx_tm88xrzhmhcfmwpj/T/ipykernel_61004/11

In [16]:
results_merged = pd.DataFrame(results)
results_merged = results_merged[results_merged.dex_key!='DexB']
results_merged = results_merged[~results_merged['sample'].str.contains('all-')]
results_merged.head(3)

,dex_key,model_config,sample,tcr_info_sequenced_only,setting,roc_auc,pr_auc,f1,precision,recall,accuracy,mcc,fdr
10,DexA,DextraDemixer,50-50-D2,True,DexA 50% (has TCR),0.999938,0.998851,0.982456,1.000000,0.965517,0.998296,0.981728,0.000000
11,DexA,BEAM,50-50-D2,True,DexA 50% (has TCR),1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
12,DexA,ICON,50-50-D2,True,DexA 50% (has TCR),NaN,NaN,0.965517,0.965517,0.965517,0.996593,0.963725,0.034483


# Store results

In [17]:
results_merged.to_csv('results/benchmark_results.csv')

In [18]:
mdata_wTCR.write('../../data/Gemund_wTCR_wPredictions.h5mu')